# 04c - ETA theo bài báo LaDe (Protocol P)

Đây là bài toán ETA chính của đồ án. Tại thời điểm shipper vừa giao xong một đơn, mô hình dự đoán số phút còn lại của các đơn đang cầm. Notebook gọi đúng hàm của Prefect task `4c`, đọc Bronze, tạo mẫu theo quy trình LaDe, huấn luyện SPEED và LightGBM, rồi ghi `eta_paper_gold` và `eta_paper_metrics_gold`.

Chạy sau `01_ingest_bronze.ipynb`; nên chạy sau `03b_star_dimensions.ipynb` để theo đúng thứ tự pipeline. Kết quả Protocol P giữ nhãn 1-120 phút theo quy trình bài báo.

In [1]:
import sys
sys.path.insert(0, '/home/jovyan/pipeline')
import steps
spark = steps.get_spark('Delivery - ETA paper Protocol P notebook')
result = steps.build_eta_paper(spark, seeds=(42, 43, 44))
result

{'run_id': '6a15165022fa4cfbbf6eb35362aac5a1',
 'raw': 206431,
 'drop_step_A': 368,
 'drop_step_C': 0,
 'kept': 206063,
 'pairs_candidate': 1915541,
 'pairs_removed_by_label_filter': 0.4816,
 'removed_label_zero': 0.0029,
 'removed_label_over_120': 0.4787,
 'pairs': {'train': 567672, 'validation': 202889, 'test': 198463}}

## Đọc metrics sau khi chạy

Bảng `eta_paper_metrics_gold` lưu MAE, RMSE và accuracy của median, SPEED và LightGBM theo từng seed.

In [2]:
metrics = spark.read.format('delta').load('s3a://lakehouse/warehouse/eta_paper_metrics_gold')
metrics.orderBy('model', 'seed', 'metric').show(100, truncate=False)

+--------------------------------+------------+----+-----------+------------------+
|run_id                          |model       |seed|metric     |value             |
+--------------------------------+------------+----+-----------+------------------+
|6a15165022fa4cfbbf6eb35362aac5a1|lightgbm    |42  |acc20      |59.147548913399476|
|6a15165022fa4cfbbf6eb35362aac5a1|lightgbm    |42  |acc30      |76.18397383895234 |
|6a15165022fa4cfbbf6eb35362aac5a1|lightgbm    |42  |mae        |20.68829787620978 |
|6a15165022fa4cfbbf6eb35362aac5a1|lightgbm    |42  |rmse_paper |24.95309314098919 |
|6a15165022fa4cfbbf6eb35362aac5a1|lightgbm    |42  |rmse_pooled|27.13181459602439 |
|6a15165022fa4cfbbf6eb35362aac5a1|lightgbm    |43  |acc20      |59.22716073021168 |
|6a15165022fa4cfbbf6eb35362aac5a1|lightgbm    |43  |acc30      |76.17893511636929 |
|6a15165022fa4cfbbf6eb35362aac5a1|lightgbm    |43  |mae        |20.671433322577247|
|6a15165022fa4cfbbf6eb35362aac5a1|lightgbm    |43  |rmse_paper |24.933773083